In [1]:
pip install pandas requests beautifulsoup4 lxml html5lib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# กำหนด path
local_path = "./owner_trade/"
local_path_data = "./owner_trade/data_owner_trade/"

In [3]:
import os
# ระบุพาธไฟล์ที่ต้องการลบ
target_file_1 = local_path_data+'owner_trade.csv'

# ตรวจสอบว่าไฟล์ 1 มีอยู่หรือไม่
if os.path.exists(target_file_1):
    os.remove(target_file_1)  # ลบไฟล์
    print(f"ลบไฟล์ {target_file_1} เรียบร้อยแล้ว")
else:
    print(f"ไม่พบไฟล์ {target_file_1} ที่ต้องการลบ")

ลบไฟล์ ./owner_trade/data_owner_trade/owner_trade.csv เรียบร้อยแล้ว


In [4]:
import pandas as pd
import requests
import datetime
import io
import time
import os
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# --- ส่วนที่ 1: กำหนด Save Path ใหม่ ---
# ใช้ r นำหน้าเพื่อบอกว่าเป็น Raw String สำหรับ Windows Path
target_folder = "./owner_trade/data_owner_trade/"

# สร้างโฟลเดอร์ถ้ายังไม่มี (รวมถึงโฟลเดอร์ย่อยทั้งหมด)
if not os.path.exists(target_folder):
    os.makedirs(target_folder)
    print(f"สร้างโฟลเดอร์สำเร็จที่: {target_folder}")

# --- ส่วนที่ 2: ตั้งค่าวันที่และ Session ---
date_to = datetime.datetime.today().strftime("%Y%m%d")
date_from = (datetime.datetime.today() - datetime.timedelta(days=5000)).strftime("%Y%m%d")

url = f"https://market.sec.or.th/public/idisc/th/Viewmore/r59-2?DateType=1&DateFrom={date_from}&DateTo={date_to}"
print(f"กำลังดึงข้อมูลจาก: {url}")

# --- ส่วนที่ 2: ตั้งค่า Session เพื่อป้องกันการโดนตัดการเชื่อมต่อ ---
session = requests.Session()
# ตั้งค่าให้มันลองใหม่ (Retry) 3 ครั้งหากการเชื่อมต่อมีปัญหา
retry_strategy = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)

# Headers ที่เลียนแบบ Chrome Browser จริงๆ
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "th-TH,th;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://market.sec.or.th/",
    "Connection": "keep-alive"
}

# --- ส่วนที่ 3: เริ่มการดึงข้อมูล ---
try:
    # ดึงข้อมูลพร้อมตั้งค่า timeout 30 วินาที
    response = session.get(url, headers=headers, timeout=30)
    response.encoding = 'utf-8-sig'
    
    if response.status_code == 200:
        # ใช้ BeautifulSoup แปลง HTML
        soup = BeautifulSoup(response.text, "html.parser")
        
        # ค้นหาตาราง
        html_io = io.StringIO(str(soup))
        tables = pd.read_html(html_io)
        
        if tables:
            df = tables[0]
            
            # --- ส่วนที่ 4: บันทึกไฟล์ลง Path ที่กำหนด ---
            file_name = f"owner_trade.csv"
            full_path = os.path.join(target_folder, file_name)
            
            df.to_csv(full_path, index=False, encoding='utf-8-sig')
            
            print("-" * 30)
            print(f"บันทึกไฟล์สำเร็จ!")
            print(f"ที่อยู่ไฟล์: {full_path}")
            print(f"จำนวนข้อมูล: {len(df)} แถว")
        else:
            print("ไม่พบตารางข้อมูลในหน้านี้")
    else:
        print(f"ไม่สามารถเข้าถึงเว็บไซต์ได้ Status Code: {response.status_code}")

except Exception as e:
    print(f"เกิดข้อผิดพลาด: {e}")

กำลังดึงข้อมูลจาก: https://market.sec.or.th/public/idisc/th/Viewmore/r59-2?DateType=1&DateFrom=20120616&DateTo=20260223
------------------------------
บันทึกไฟล์สำเร็จ!
ที่อยู่ไฟล์: ./owner_trade/data_owner_trade/owner_trade.csv
จำนวนข้อมูล: 85868 แถว
